In [1]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
from transformers import pipeline
import textstat

# Load both evaluation files
dialog_df = pd.read_csv("/kaggle/input/genmedqa-100/dialog_test_results.csv")   # Update with actual path
qa_df = pd.read_csv("/kaggle/input/genmedqa-100/qa_test_results.csv")           # Update with actual path

# Load pretrained sentiment and entailment models
sentiment_classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
nli_model = pipeline("text-classification", model="roberta-large-mnli")

2025-04-22 02:06:00.546190: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745287560.964200      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745287561.085678      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [3]:
def extract_generated_doctor_text(text):
    if "Doctor:" in text:
        return text.split("Doctor:")[-1].strip()
    return text.strip()

# Apply to both dialog and QA files
dialog_df["gen_doctor"] = dialog_df["generated_answer"].apply(extract_generated_doctor_text)
qa_df["gen_doctor"] = qa_df["generated_answer"].apply(extract_generated_doctor_text)

In [4]:
def get_sentiment_label(text):
    try:
        return sentiment_classifier(text)[0]['label']
    except:
        return "ERROR"

def get_entailment(premise, hypothesis):
    try:
        return nli_model(f"{premise} </s> {hypothesis}")[0]['label']
    except:
        return "ERROR"

def get_readability(text):
    return {
        "Flesch Reading Ease": textstat.flesch_reading_ease(text),
        "Grade Level": textstat.flesch_kincaid_grade(text),
        "Gunning Fog": textstat.gunning_fog(text)
    }

In [5]:
def evaluate(df):
    df["sentiment_true"] = df["true_answer"].apply(get_sentiment_label)
    df["sentiment_gen"] = df["gen_doctor"].apply(get_sentiment_label)
    df["sentiment_match"] = df["sentiment_true"] == df["sentiment_gen"]

    df["entailment"] = df.apply(
        lambda row: get_entailment(row["question"], row["gen_doctor"]), axis=1
    )

    readability = df["gen_doctor"].apply(get_readability).apply(pd.Series)
    df = pd.concat([df, readability], axis=1)

    return df

In [6]:
dialog_eval = evaluate(dialog_df)
qa_eval = evaluate(qa_df)

Token indices sequence length is longer than the specified maximum sequence length for this model (871 > 512). Running this sequence through the model will result in indexing errors
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [7]:
def summarize(df, label):
    sentiment_rate = df["sentiment_match"].mean()
    entailment_rate = (df["entailment"] == "ENTAILMENT").mean()
    readability_avg = df[["Flesch Reading Ease", "Grade Level", "Gunning Fog"]].mean()

    print(f"\n{label} Evaluation Summary")
    print("-" * 40)
    print(f"Sentiment Match Rate: {sentiment_rate:.2%}")
    print(f"Entailment Rate (Factuality): {entailment_rate:.2%}")
    print("Readability Scores (Avg):")
    print(readability_avg)

summarize(dialog_eval, "Dialog Model")
summarize(qa_eval, "QA Model")


Dialog Model Evaluation Summary
----------------------------------------
Sentiment Match Rate: 50.00%
Entailment Rate (Factuality): 2.00%
Readability Scores (Avg):
Flesch Reading Ease    63.5177
Grade Level             7.7150
Gunning Fog             8.8469
dtype: float64

QA Model Evaluation Summary
----------------------------------------
Sentiment Match Rate: 80.00%
Entailment Rate (Factuality): 2.00%
Readability Scores (Avg):
Flesch Reading Ease    54.9796
Grade Level             9.2140
Gunning Fog            10.3878
dtype: float64
